# Signalling-Bots on Colab

1. **Runtime → Change runtime type → GPU** (e.g. T4).
2. Run the cells in order. All seeds in one run are evolved together on the GPU (`vmap` over seeds), so 10 seeds cost about the same as 1.
3. Results go to `results/<experiment>/s<seed>_g<generations>/` and are copied to Google Drive (or downloaded as a zip) at the end.

GPU maths rounds slightly differently from the Mac CPU, so a seed run here won't match the same seed run locally. Use new seed numbers to keep them apart.

In [ ]:
!nvidia-smi -L
import jax
print("jax", jax.__version__, jax.devices())

In [ ]:
REPO = "https://github.com/migarak2908/Signalling-Bots"
BRANCH = "main"

import os
os.chdir("/content")
if not os.path.exists("Signalling-Bots"):
    !git clone -b {BRANCH} {REPO}
os.chdir("/content/Signalling-Bots")
!git pull
!git log --oneline -1

In [ ]:
# Experiments: 2col_comm, 2col_nocomm, 3col_comm, 3col_nocomm
EXPERIMENT = "3col_comm"
SEEDS = "10 11 12 13 14 15 16 17 18 19"
GENERATIONS = 3000

!python -m evolve {EXPERIMENT} --seeds {SEEDS} --generations {GENERATIONS}

In [ ]:
!python -m analysis.summarize_runs {EXPERIMENT}

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for seed in SEEDS.split():
    run = Path("results") / EXPERIMENT / f"s{seed}_g{GENERATIONS}"
    for plot in ("fitness.png", "trajectories.png"):
        display(Image(str(run / plot), width=900))

In [ ]:
# Save results: copy into Google Drive (MyDrive/Signalling-Bots/results), or download a zip.
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/Signalling-Bots/results
    !cp -r results/{EXPERIMENT} /content/drive/MyDrive/Signalling-Bots/results/
    print("copied to MyDrive/Signalling-Bots/results/" + EXPERIMENT)
else:
    from google.colab import files
    !zip -qr {EXPERIMENT}.zip results/{EXPERIMENT}
    files.download(f"{EXPERIMENT}.zip")